# 1. Config & Imports

In [2]:

from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    audio_root : str = "/kaggle/input/datasets/mishbhaul/indian-raga-dataset/RagaDataset/Carnatic/audio"
    features_csv : str = "mfcc_chroma_features.csv"

    # audio settings
    sr : int = 22050
    analysis_duration : float | None = None # considered full clip
    fmin_hz : float = 80.0
    fmax_hz : float = 800.0
    n_mfcc : int = 13              # standard MFCC coefficient count
    use_mfcc_delta : bool = True   # also include first-derivative (delta) MFCCs
    n_chroma : int = 12            # 12 = one bin per semitone
    align_chroma_to_tonic : bool = True  # circularly shift chroma so tonic bin -> index 0

    random_seed : int = 42
    AUDIO_EXTENSIONS = (".wav", ".mp3", ".flac", ".m4a")
cfg = Config()


In [3]:

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMBA_NUM_THREADS", "1")

import re
import warnings

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from joblib import Parallel, delayed

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(cfg.random_seed)


### Helper functions

In [4]:

# Raga IDs for 300 Hindustani Classical Music clips (30 ragas x 10 songs)
ID_TO_RAGA = {
    'db085e26-665d-4f4c-a2a5-95251c22b69e': 'Pūrvīkaḷyāṇi',
    '2165542c-45da-4301-af82-c2e7ddbe4768': 'Bilahari',
    '123b09bd-9901-4e64-a65a-10b02c9e0597': 'Bhairavi',
    'df85a0a5-b1a8-42f1-a87b-3d7c7ee33fb4': 'Kēdāragauḷa',
    'cda9cbe9-c1aa-42bb-8b4c-e9dfc8af133c': 'Nāṭakurinji',
    '839bb6b4-1577-4a78-9e5d-4906c6453274': 'Suraṭi',
    'a9413dff-91d1-4e29-ad92-c04019dce5b8': 'Tōḍi',
    '5ce23030-f71d-4f5d-9d76-f91c2c182392': 'Dhanyāsi',
    '978f4c3c-6a12-43fd-8427-c414ee17b252': 'Mukhāri',
    'a4ec6633-1050-4207-8313-b017194c8fa0': 'Kamās',
    'e8a0bf54-13c6-4a09-922a-bfc744ddf38a': 'Varāḷi',
    'c6b5f8d9-ebb4-46af-a020-6646fce2c77d': 'Sāma',
    '993d6cf6-dc89-4d23-9a9f-8eeed1524872': 'Sahānā',
    'e18fcaa7-1f9c-4f09-b627-0687481f4ec7': 'Ānandabhairavi',
    '9cedca68-4a9d-4170-bec3-0d1db1ff730e': 'Māyāmāḷavagauḷa',
    'bdd80890-44f1-4a93-8d93-06a418781f97': 'Śrī',
    'bf4662d4-25c3-4cad-9249-4de2dc513c06': 'Kalyāṇi',
    '50bd048f-4482-4c5b-850c-9ad5e5ec46f1': 'Hussēnī',
    '9a071e54-3eed-48b2-83a3-1a3fd683b8e0': 'Kānaḍa',
    '42dd0ccb-f92a-4622-ae5d-a3be571b4939': 'Śrīranjani',
    '6345e8fe-7061-4bdc-842c-dcfd4a379820': 'Aṭāna',
    '98d46d9e-5100-4f24-bcd4-d0f95966c7cb': 'Sindhubhairavi',
    '18b1acb9-dff6-47ec-873a-b2086c8d268d': 'Madhyamāvati',
    'defad4cc-48aa-4372-a31a-c43624930713': 'Dēvagāndhāri',
    'f0866e71-33b2-47db-ab75-0808c41f2401': 'Kāṁbhōji',
    'a47e9d22-847a-46e8-b589-2a3537789f5f': 'Sāvēri',
    '09c179f3-8b19-4792-a852-e9fa0090e409': 'Kāpi',
    '85ccf631-4cdf-4f6c-a841-0edfcf4255d1': 'Kāmavardani',
    'aa5f376f-06cd-4a69-9cc9-b1077104e0b0': 'Bēgaḍa',
    'e5c4d94a-b34a-42ef-acd7-f235612350e4': 'Gauḷa',
    'a2f9f182-0ceb-4531-b286-b840b47a54b8': 'Śankarābharaṇaṁ',
    '4ce0b18d-f4df-41cf-9b40-9166199506b4': 'Rītigauḷa',
    '700e1d92-7094-4c21-8a3b-d7b60c550edf': 'Behāg',
    '9ebeb536-30a7-403f-8042-f7c1445c4b87': 'Nāṭa',
    '39821826-3327-41d7-9cd5-e22fe7b08360': 'Mōhanaṁ',
    '0277eae5-3411-4b22-9fa8-1b347e7528d1': 'Ṣanmukhapriya',
    'b08475a2-1049-433a-be8f-105bf14718fb': 'Harikāmbhōji',
    'ddff55ae-20f6-4d7d-ba9e-a6c10eeebd41': 'Yadukula kāṁbōji',
    'f972db4d-5d16-4f9a-9841-f313e1601aaa': 'Karaharapriya',
    '3af5a361-923a-465d-864d-9c7ba0c04a47': 'Sencuruṭṭi'
}

print(f"Loaded {len(ID_TO_RAGA)} raga id mappings "
      f"({len(set(ID_TO_RAGA.values()))} unique raga labels)")


def clean_uuid(raw_name: str) -> str:
    '''Normalize a folder name to its canonical uuid string.'''
    name = raw_name.strip()
    name = re.sub(r"\s*\(\d+\)$", "", name)   # trailing " (1)", " (2)"
    name = re.sub(r"\s+\d+$", "", name)       # trailing " 2", " 3"
    return name.lower()


def build_normalized_lookup(id_to_raga: dict) -> dict:
    '''Collapse duplicate-suffixed uuid keys into one clean lookup table.'''
    return {clean_uuid(k): v for k, v in id_to_raga.items()}


def find_audio_files_with_labels(root_dir: str, id_to_raga: dict) -> list[tuple[str, str, str]]:
    '''Walk `root_dir` and return (filepath, raga_label, artist) for every
    audio file whose parent uuid folder is present in `id_to_raga`.
    '''
    lookup = build_normalized_lookup(id_to_raga)
    items = []

    for uuid_folder in sorted(os.listdir(root_dir)):
        uuid_path = os.path.join(root_dir, uuid_folder)
        if not os.path.isdir(uuid_path):
            continue

        raga_label = lookup.get(clean_uuid(uuid_folder))
        if raga_label is None:
            print(f"WARNING: no raga mapping for uuid folder '{uuid_folder}', skipping")
            continue

        for dirpath, _, filenames in os.walk(uuid_path):
            rel_path = os.path.relpath(dirpath, uuid_path)
            artist = rel_path.split(os.sep)[0] if rel_path != "." else "unknown"

            for fname in filenames:
                if fname.lower().endswith(cfg.AUDIO_EXTENSIONS):
                    items.append((os.path.join(dirpath, fname), raga_label, artist))

    return items


Loaded 40 raga id mappings (40 unique raga labels)



# 2. Feature Extraction

### 2.1 Tonic estimation

Used here only to **align chroma to the tonic** (Section 2.3) — MFCCs
don't need it, since timbre isn't pitch-relative.


In [5]:

# "Cents" is just a unit for measuring musical pitch on a logarithmic scale
# 1200 cents = exactly 1 octave --> 100 cents = 1 semitone (12 semitones in an octave)
def estimate_tonic(audio : np.ndarray, sr : int, low_freq : float = cfg.fmin_hz, high_freq : float = cfg.fmax_hz):
    pitch_track, is_voiced, _ = librosa.pyin(
        audio, fmin=low_freq, fmax=high_freq, sr=sr, frame_length=2048
    )
    # keep only the pitch estimates where a clear pitch was actually detected
    if is_voiced is not None:
        clean_pitches = pitch_track[is_voiced]
    else:
        clean_pitches = pitch_track[~np.isnan(pitch_track)]
    clean_pitches = clean_pitches[~np.isnan(clean_pitches)]

    if len(clean_pitches) == 0:
        return 220.0  # no pitch detected in this clip, just return a default

    # convert to cents
    cents = 1200*np.log2(clean_pitches / low_freq)
    cents_in_one_octave = np.mod(cents, 1200)
    counts, bin_edges = np.histogram(cents_in_one_octave, bins=120, range=(0,1200))
    tallest_bin_position = bin_edges[np.argmax(counts)]

    # convert that cents value back into a frequency in Hz
    tonic_hz = low_freq * (2 ** (tallest_bin_position / 1200))
    return float(tonic_hz)


def tonic_to_chroma_bin(tonic_hz: float, n_chroma: int = cfg.n_chroma) -> int:
    '''Map a tonic frequency to the nearest chroma bin index (0 = C by librosa convention).'''
    semitones_from_c = 12 * np.log2(tonic_hz / 261.63)
    return int(round(semitones_from_c)) % n_chroma



### 2.2 MFCC extraction

Captures the *timbral* texture of the clip (spectral envelope shape).
We time-pool (mean + std across frames) each coefficient into a fixed-
length vector, plus optional delta (first-derivative) coefficients to
capture how the timbre changes over time.


In [6]:

def extract_mfcc_features(y: np.ndarray, sr: int, n_mfcc: int = cfg.n_mfcc,
                           use_delta: bool = cfg.use_mfcc_delta) -> dict:
    '''Compute MFCCs (+ deltas), time-pooled to mean/std per coefficient.'''
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    feat = {}
    for i in range(n_mfcc):
        feat[f"mfcc_{i}_mean"] = float(np.mean(mfcc[i]))
        feat[f"mfcc_{i}_std"] = float(np.std(mfcc[i]))

    if use_delta:
        delta = librosa.feature.delta(mfcc)
        for i in range(n_mfcc):
            feat[f"mfcc_delta_{i}_mean"] = float(np.mean(delta[i]))
            feat[f"mfcc_delta_{i}_std"] = float(np.std(delta[i]))

    return feat



### 2.3 Chroma extraction

Captures *pitch-class* energy — the off-the-shelf, spectrogram-based
analogue of a PCD, computed directly via CQT rather than from onset-
segmented notes. Since Hindustani "Sa" is a movable tonic but chroma
bins are absolute pitch (C, C#, D, ...), we optionally roll the chroma
vector so the tonic bin lands at index 0, using the tonic estimated in
2.1 — this makes clips in different keys comparable.


In [7]:

def extract_chroma_features(y: np.ndarray, sr: int, tonic_hz: float | None = None,
                             n_chroma: int = cfg.n_chroma,
                             align_to_tonic: bool = cfg.align_chroma_to_tonic) -> dict:
    '''Compute chroma-CQT, optionally tonic-aligned, time-pooled to mean/std per bin.'''
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, n_chroma=n_chroma)

    if align_to_tonic and tonic_hz is not None:
        shift = tonic_to_chroma_bin(tonic_hz, n_chroma)
        chroma = np.roll(chroma, -shift, axis=0)  # tonic bin -> index 0

    feat = {}
    for i in range(n_chroma):
        feat[f"chroma_{i}_mean"] = float(np.mean(chroma[i]))
        feat[f"chroma_{i}_std"] = float(np.std(chroma[i]))

    return feat


# 3. Build Feature Table

In [8]:

def process_file(filepath: str, raga_label: str, artist: str,
                  sr_target: int = cfg.sr,
                  duration: float | None = cfg.analysis_duration) -> dict:
    '''Load one clip and compute its MFCC + chroma feature row.'''
    y, sr = librosa.load(filepath, sr=sr_target, duration=duration, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)

    tonic_hz = estimate_tonic(y, sr) if cfg.align_chroma_to_tonic else None
    mfcc_feat = extract_mfcc_features(y, sr)
    chroma_feat = extract_chroma_features(y, sr, tonic_hz=tonic_hz)

    row = {"filepath": filepath, "artist": artist, "raga": raga_label,
           "tonic_hz": tonic_hz if tonic_hz is not None else np.nan}
    row.update(mfcc_feat)
    row.update(chroma_feat)
    return row


def _safe_process_file(filepath: str, raga_label: str, artist: str) -> dict | None:
    '''Wrap process_file so one bad/corrupt clip returns None instead of
    raising inside a worker.'''
    try:
        return process_file(filepath, raga_label, artist)
    except Exception as e:
        print(f"FAILED  {filepath}: {e}")
        return None


def safe_filename(name: str) -> str:
    '''Turn a raga label into a filesystem-safe filename.'''
    return re.sub(r"[^\w\-]+", "_", name.strip()).strip("_").lower()


def build_feature_table_for_raga(raga_name: str, root: str, id_to_raga: dict,
                                  out_dir: str = "raga_features",
                                  force: bool = False,
                                  n_jobs: int = 4) -> pd.DataFrame:
    '''Extract MFCC + chroma features for a single raga only, in parallel
    across n_jobs CPU cores, caching the result to its own CSV.'''
    os.makedirs(out_dir, exist_ok=True)
    raga_csv = os.path.join(out_dir, f"{safe_filename(raga_name)}.csv")

    if os.path.exists(raga_csv) and not force:
        print(f"Loading cached features for '{raga_name}' from {raga_csv}")
        return pd.read_csv(raga_csv)

    all_files = find_audio_files_with_labels(root, id_to_raga)
    raga_files = [(fp, label, artist) for fp, label, artist in all_files if label == raga_name]
    if not raga_files:
        raise ValueError(f"No audio files found for raga '{raga_name}'")

    print(f"Found {len(raga_files)} audio files for raga '{raga_name}' (n_jobs={n_jobs})")

    results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_safe_process_file)(filepath, raga_label, artist)
        for filepath, raga_label, artist in tqdm(raga_files, desc=f"Extracting: {raga_name}")
    )

    rows = [r for r in results if r is not None]
    if len(rows) < len(raga_files):
        print(f"WARNING: {len(raga_files) - len(rows)} file(s) failed for '{raga_name}'")

    df = pd.DataFrame(rows)
    df.to_csv(raga_csv, index=False)
    print(f"Saved {len(df)} rows to {raga_csv}")
    return df


def combine_all_raga_features(out_dir: str = "raga_features",
                               out_csv: str = cfg.features_csv) -> pd.DataFrame:
    '''Once you've run build_feature_table_for_raga on all ragas you want,
    stitch their per-raga CSVs together into one combined feature table.'''
    csv_files = sorted(f for f in os.listdir(out_dir) if f.endswith(".csv"))
    if not csv_files:
        raise FileNotFoundError(f"No per-raga CSVs found in {out_dir}")

    dfs = [pd.read_csv(os.path.join(out_dir, f)) for f in csv_files]
    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(out_csv, index=False)

    n_ragas = combined["raga"].nunique() if len(combined) else 0
    print(f"Combined {len(csv_files)} raga CSV(s) -> {len(combined)} rows -> {out_csv}")
    print(f"Ragas represented: {n_ragas}")
    return combined



### Combine per-raga results

Run once you've extracted features for all the ragas you need.


In [9]:
raga_to_process = "Ṣanmukhapriya"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Ṣanmukhapriya' (n_jobs=4)


Extracting: Ṣanmukhapriya:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/ṣanmukhapriya.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Ṣanmukhapriya,145.875598,-198.480286,61.826797,103.283813,22.653917,-17.908749,16.457134,...,0.554566,0.332166,0.344229,0.308391,0.319870,0.259021,0.331947,0.273730,0.318643,0.250615
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Kunnakkudi_M_Balamuralikrishna,Ṣanmukhapriya,140.906540,-164.792465,76.973083,115.503609,26.606411,-11.489194,15.088002,...,0.546790,0.320294,0.442426,0.293539,0.357052,0.260978,0.369524,0.267383,0.342550,0.245341
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,S__Sowmya,Ṣanmukhapriya,99.635970,-167.481186,48.191158,104.993919,23.524977,-23.855389,18.476385,...,0.532055,0.307846,0.400960,0.269924,0.341157,0.230607,0.410775,0.269793,0.323250,0.219091
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Ṣanmukhapriya,145.875598,-141.578949,62.047710,114.275864,19.851305,-43.514942,19.274397,...,0.639800,0.292448,0.411315,0.253183,0.413159,0.257025,0.435055,0.274524,0.394472,0.238997
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T_K_Rangachari,Ṣanmukhapriya,139.288090,-275.835815,27.650045,205.196930,22.837854,23.945700,11.386627,...,0.585436,0.329436,0.339921,0.261600,0.372430,0.256374,0.414934,0.306881,0.298212,0.220535


In [11]:
raga_to_process = "Pūrvīkaḷyāṇi"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Pūrvīkaḷyāṇi' (n_jobs=4)


Extracting: Pūrvīkaḷyāṇi:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/pūrvīkaḷyāṇi.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Pūrvīkaḷyāṇi,87.240619,-208.971771,64.749191,133.546677,23.698954,-21.005301,19.170994,...,0.516056,0.304985,0.358143,0.230695,0.404442,0.297043,0.320928,0.222247,0.325801,0.217977
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Pūrvīkaḷyāṇi,85.248040,-164.334061,65.478661,98.826881,20.354881,-25.034328,18.001547,...,0.623896,0.270788,0.409487,0.237791,0.432083,0.262560,0.394277,0.227080,0.487213,0.252735
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Kunnakkudi_M_Balamuralikrishna,Pūrvīkaḷyāṇi,140.906540,-222.976669,76.229927,116.796860,24.847626,-12.596081,15.925631,...,0.437408,0.299898,0.287554,0.227569,0.282302,0.278225,0.297178,0.270131,0.304711,0.234871
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Pūrvīkaḷyāṇi,146.720647,-286.937500,67.645584,117.516617,24.848030,-12.541356,19.176611,...,0.669631,0.275507,0.444503,0.259563,0.434938,0.293407,0.394859,0.245435,0.436501,0.256119
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Pūrvīkaḷyāṇi,146.720647,-231.651398,90.035530,107.320152,27.304285,-17.165913,23.890572,...,0.612603,0.302862,0.324083,0.246800,0.351527,0.285826,0.382726,0.268501,0.411371,0.266760


In [12]:
raga_to_process = "Bilahari"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Bilahari' (n_jobs=4)


Extracting: Bilahari:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/bilahari.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Unnikrishnan,Bilahari,130.713236,-121.099464,60.137535,81.755379,29.964333,-57.014606,16.677500,...,0.452589,0.308583,0.252979,0.215911,0.402089,0.316922,0.324952,0.266725,0.330680,0.240121
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Amrutha_Venkatesh,Bilahari,98.491553,-245.181503,65.820229,90.742149,28.407024,-24.544895,23.248991,...,0.377877,0.305367,0.200569,0.210695,0.285739,0.291443,0.287127,0.262040,0.314944,0.270642
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Bilahari,145.875598,-174.584366,66.538475,118.012367,24.197338,-5.168667,17.428556,...,0.617339,0.288515,0.433126,0.264887,0.468171,0.301919,0.455816,0.268712,0.455034,0.246502
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sumithra_Vasudev,Bilahari,97.924283,-179.582321,67.966103,102.120972,22.730520,-15.033888,20.158506,...,0.555431,0.289601,0.413645,0.226622,0.409821,0.242622,0.431357,0.240459,0.426301,0.232912
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Ranjani_Guruprasad,Bilahari,103.747164,-212.861694,78.004585,111.391815,25.566164,-25.740675,23.311789,...,0.537938,0.310434,0.377904,0.209628,0.445471,0.251670,0.449204,0.232836,0.617898,0.263812


In [13]:
raga_to_process = "Kēdāragauḷa"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Kēdāragauḷa' (n_jobs=4)


Extracting: Kēdāragauḷa:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/kēdāragauḷa.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Unnikrishnan,Kēdāragauḷa,138.485850,-137.827667,61.109947,98.309616,21.590868,-33.458206,14.351354,...,0.516165,0.284805,0.393067,0.247242,0.479712,0.284570,0.427339,0.246333,0.423271,0.258277
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Kēdāragauḷa,145.875598,-227.426926,89.683708,97.443001,28.218651,-6.162536,23.552170,...,0.533019,0.315538,0.346675,0.269379,0.393129,0.284022,0.397524,0.281580,0.424476,0.270856
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Kēdāragauḷa,145.875598,-256.281738,91.375862,80.156631,30.670525,-40.031437,21.870134,...,0.553614,0.270893,0.405982,0.230042,0.429574,0.254526,0.422425,0.239130,0.507040,0.247578
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Kēdāragauḷa,145.875598,-232.646225,63.402740,74.309860,30.443670,-51.048439,20.592598,...,0.460533,0.321187,0.234333,0.192701,0.278151,0.243639,0.259174,0.233579,0.338785,0.256738
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Nithyasree_Mahadevan,Kēdāragauḷa,108.027996,-236.838669,96.980049,99.466286,20.641207,-26.222479,20.142162,...,0.492287,0.272409,0.342287,0.208984,0.380693,0.225638,0.496938,0.271456,0.609190,0.288730


In [14]:
raga_to_process = "Nāṭakurinji"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Nāṭakurinji' (n_jobs=4)


Extracting: Nāṭakurinji:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/nāṭakurinji.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Nāṭakurinji,145.875598,-184.279587,63.965683,111.660210,21.025410,-13.217031,17.250385,...,0.489987,0.272557,0.391849,0.249907,0.505292,0.329667,0.380511,0.264772,0.352209,0.222374
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Nāṭakurinji,146.720647,-195.499832,76.429817,110.989517,27.563284,-11.402169,15.693097,...,0.390126,0.256365,0.339755,0.250296,0.474071,0.351357,0.423047,0.302249,0.360370,0.239823
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Nāṭakurinji,145.875598,-205.025162,72.531937,97.920845,26.128942,-6.225230,22.727829,...,0.426885,0.252352,0.343129,0.251155,0.496048,0.352279,0.348350,0.263421,0.354259,0.247227
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Amritha_Murali,Nāṭakurinji,97.924283,-243.236130,49.239006,93.970123,22.280516,-12.363018,15.962483,...,0.287500,0.210466,0.230900,0.192304,0.444239,0.342532,0.389582,0.293084,0.329892,0.237054
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Amritha_Murali,Nāṭakurinji,99.062109,-188.394272,56.869549,82.014053,28.464649,-17.365246,18.394030,...,0.307479,0.221046,0.273311,0.213101,0.381987,0.330366,0.365687,0.276407,0.337680,0.247966


In [15]:
raga_to_process = "Suraṭi"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Suraṭi' (n_jobs=4)


Extracting: Suraṭi:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/suraṭi.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Suraṭi,145.875598,-229.108368,68.070740,104.733696,24.479578,-14.514230,20.124252,...,0.570901,0.308899,0.339180,0.268422,0.397508,0.268834,0.420135,0.296538,0.306060,0.205336
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Kunnakkudi_M_Balamuralikrishna,Suraṭi,139.288090,-178.491913,66.107887,115.181648,23.303732,-16.944952,18.679798,...,0.536617,0.293246,0.454247,0.260924,0.457660,0.275712,0.548762,0.305084,0.478073,0.291756
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Suraṭi,109.916292,-202.613937,87.496262,101.446358,26.760622,-21.754524,19.540222,...,0.431389,0.315268,0.287984,0.266549,0.360198,0.291202,0.336351,0.313547,0.247938,0.232360
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Suraṭi,109.916292,-157.427811,66.096916,89.736282,22.776989,-27.989244,19.647076,...,0.375934,0.267882,0.342925,0.268262,0.417593,0.277321,0.491351,0.314540,0.362207,0.255544
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Nithyasree_Mahadevan,Suraṭi,108.027996,-189.047989,70.979332,94.302490,28.934526,-41.477623,19.499199,...,0.465811,0.287765,0.285318,0.207984,0.417683,0.264689,0.474362,0.272901,0.452705,0.252309


In [16]:
raga_to_process = "Ānandabhairavi"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Ānandabhairavi' (n_jobs=4)


Extracting: Ānandabhairavi:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/ānandabhairavi.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Ānandabhairavi,84.757048,-176.497620,65.300903,102.211227,22.910192,-16.387159,18.852564,...,0.452699,0.285139,0.407137,0.264280,0.320769,0.236524,0.352768,0.231942,0.301434,0.207118
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Ānandabhairavi,85.248040,-172.452957,59.228996,112.230576,23.282858,-12.628272,17.346806,...,0.590474,0.308765,0.327266,0.241626,0.334432,0.235725,0.288781,0.212905,0.418285,0.259775
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Maharajapuram_S_Ramachandran,Ānandabhairavi,122.666268,-192.663422,48.792198,126.687286,25.189291,-10.580125,15.863123,...,0.543182,0.309420,0.407014,0.258625,0.414146,0.268125,0.372073,0.262125,0.333473,0.230905
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Ānandabhairavi,109.916292,-218.465759,67.852325,69.273300,36.232841,-6.466208,20.073345,...,0.382370,0.310165,0.293617,0.274432,0.333211,0.262673,0.382840,0.334052,0.275555,0.231462
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Amritha_Murali,Ānandabhairavi,97.924283,-209.910507,51.789642,83.294106,26.035204,-34.885536,19.602043,...,0.524914,0.343876,0.188986,0.180970,0.243458,0.222329,0.289923,0.245221,0.304414,0.248972


In [17]:
raga_to_process = "Karaharapriya"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Karaharapriya' (n_jobs=4)


Extracting: Karaharapriya:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/karaharapriya.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Karaharapriya,84.757048,-201.001709,80.581741,99.484795,27.833124,-21.485304,22.355799,...,0.359675,0.234547,0.511481,0.302268,0.468653,0.301311,0.540679,0.294211,0.360564,0.234415
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Karaharapriya,145.875598,-182.218048,50.359886,110.948410,21.929697,-15.778233,16.376207,...,0.621969,0.322027,0.395284,0.268875,0.462973,0.313799,0.331593,0.249697,0.261552,0.190395
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Karaharapriya,145.875598,-253.346161,86.706497,123.266769,22.711073,2.013729,23.047132,...,0.600168,0.268773,0.431704,0.283255,0.494824,0.303898,0.456217,0.279935,0.418202,0.236070
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,R_Vedavalli,Karaharapriya,140.906540,-184.828445,68.859856,114.355728,28.308897,-18.952353,20.844675,...,0.498619,0.280151,0.379434,0.239070,0.373304,0.253793,0.420027,0.262549,0.412965,0.269300
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Karaharapriya,146.720647,-223.517776,74.893349,89.518776,24.865507,-12.548841,23.798803,...,0.511260,0.324330,0.317243,0.240954,0.420876,0.311564,0.355674,0.287781,0.285108,0.228299


In [18]:
raga_to_process = "Rītigauḷa"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Found 12 audio files for raga 'Rītigauḷa' (n_jobs=4)


Extracting: Rītigauḷa:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/rītigauḷa.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Amrutha_Venkatesh,Rītigauḷa,98.491553,-238.474625,55.160427,87.016647,26.675489,-22.895004,20.264023,...,0.365769,0.262818,0.250669,0.225114,0.365449,0.299796,0.384675,0.306218,0.303489,0.245697
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Rītigauḷa,85.248040,-175.152832,70.333672,82.518448,27.629107,-22.020945,16.627348,...,0.459061,0.271631,0.344593,0.246363,0.466971,0.309818,0.386170,0.267177,0.458613,0.259887
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Kunnakkudi_M_Balamuralikrishna,Rītigauḷa,139.288090,-190.284103,72.651886,105.564919,22.892303,-21.124474,17.330606,...,0.323580,0.223907,0.333626,0.248414,0.374096,0.274008,0.404093,0.299632,0.269927,0.220148
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Rītigauḷa,146.720647,-197.729767,65.566193,112.911705,25.543848,-11.866860,16.258869,...,0.416577,0.256207,0.321202,0.229301,0.404153,0.278360,0.437923,0.300683,0.357588,0.239834
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Rītigauḷa,145.875598,-210.078751,77.330261,110.493645,23.244463,-29.074707,21.187407,...,0.453072,0.260413,0.397078,0.266728,0.457959,0.285966,0.447683,0.280281,0.422110,0.260493


In [19]:
raga_to_process = "Karaharapriya"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()


Loading cached features for 'Karaharapriya' from raga_features/karaharapriya.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Karaharapriya,84.757048,-201.001709,80.581741,99.484795,27.833124,-21.485304,22.355799,...,0.359675,0.234547,0.511481,0.302268,0.468653,0.301311,0.540679,0.294211,0.360564,0.234415
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Karaharapriya,145.875598,-182.218048,50.359886,110.948410,21.929697,-15.778233,16.376207,...,0.621969,0.322027,0.395284,0.268875,0.462973,0.313799,0.331593,0.249697,0.261552,0.190395
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Sikkil_Gurucharan,Karaharapriya,145.875598,-253.346161,86.706497,123.266769,22.711073,2.013729,23.047132,...,0.600168,0.268773,0.431704,0.283255,0.494824,0.303898,0.456217,0.279935,0.418202,0.236070
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,R_Vedavalli,Karaharapriya,140.906540,-184.828445,68.859856,114.355728,28.308897,-18.952353,20.844675,...,0.498619,0.280151,0.379434,0.239070,0.373304,0.253793,0.420027,0.262549,0.412965,0.269300
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__M__Krishna,Karaharapriya,146.720647,-223.517776,74.893349,89.518776,24.865507,-12.548841,23.798803,...,0.511260,0.324330,0.317243,0.240954,0.420876,0.311564,0.355674,0.287781,0.285108,0.228299


In [21]:
raga_to_process = "Sāvēri"

raga_df = build_feature_table_for_raga(raga_to_process, cfg.audio_root, ID_TO_RAGA, n_jobs=4)
raga_df.head()

Found 12 audio files for raga 'Sāvēri' (n_jobs=4)


Extracting: Sāvēri:   0%|          | 0/12 [00:00<?, ?it/s]

Saved 12 rows to raga_features/sāvēri.csv


,filepath,artist,raga,tonic_hz,mfcc_0_mean,mfcc_0_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,...,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std
0,/kaggle/input/datasets/mishbhaul/indian-raga-d...,T__Brinda,Sāvēri,95.136569,-151.937424,67.319923,148.591324,32.476612,-90.080399,22.732323,...,0.637863,0.312624,0.510476,0.296048,0.256084,0.233340,0.221544,0.206717,0.305674,0.245061
1,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Aruna_Sairam,Sāvēri,87.240619,-255.785370,73.059776,100.134697,26.134930,-39.490387,24.557066,...,0.611896,0.307190,0.361521,0.281644,0.289744,0.233821,0.266478,0.209659,0.330381,0.231949
2,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Prasanna_Venkataraman,Sāvēri,109.916292,-190.444656,61.890976,111.973053,23.095846,-12.719995,19.026701,...,0.317537,0.188680,0.307330,0.196586,0.452739,0.246452,0.494656,0.283556,0.421304,0.227034
3,/kaggle/input/datasets/mishbhaul/indian-raga-d...,Kuldeep_Pai,Sāvēri,145.875598,-288.192749,72.798607,104.010162,26.031696,-15.831986,21.097710,...,0.557695,0.342722,0.348362,0.309222,0.249975,0.257336,0.201222,0.190650,0.248622,0.207740
4,/kaggle/input/datasets/mishbhaul/indian-raga-d...,S__Sowmya,Sāvēri,98.491553,-163.141403,61.915009,102.492668,24.926353,-17.495090,17.736645,...,0.616524,0.291818,0.391158,0.254560,0.312669,0.202476,0.323354,0.192842,0.407699,0.219059


In [ ]:

# Once you've processed all the ragas you care about above (re-running
# cell above with a new `raga_to_process` each time), run this to build
# the single combined table the EDA section below expects.
if os.path.exists(cfg.features_csv):
    print(f"Loading cached combined features from {cfg.features_csv}")
    features_df = pd.read_csv(cfg.features_csv)
else:
    features_df = combine_all_raga_features()

features_df.head()


# 4. EDA

In [ ]:

print(f"Total clips: {len(features_df)}")
print(f"Unique ragas: {features_df['raga'].nunique()}")
print(f"Unique artists: {features_df['artist'].nunique()}")

fig, ax = plt.subplots(figsize=(10, 6))
features_df["raga"].value_counts().sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of clips")
ax.set_title("Clips per raga — check this is roughly balanced")
plt.tight_layout()
plt.show()


In [ ]:

mfcc_cols = [c for c in features_df.columns if c.startswith("mfcc_")]
chroma_cols = [c for c in features_df.columns if c.startswith("chroma_")]

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
example = features_df.sample(1, random_state=cfg.random_seed).iloc[0]

mfcc_mean_cols = [c for c in mfcc_cols if c.endswith("_mean") and "delta" not in c]
axes[0].bar(range(len(mfcc_mean_cols)), example[mfcc_mean_cols].values, color="darkslateblue")
axes[0].set_title(f"MFCC means — {example['raga']} ({example['artist']})")
axes[0].set_xlabel("MFCC coefficient")

chroma_mean_cols = [c for c in chroma_cols if c.endswith("_mean")]
axes[1].bar(range(len(chroma_mean_cols)), example[chroma_mean_cols].values, color="darkorange", width=1.0)
axes[1].set_title("Chroma means (tonic-aligned, bin 0 = tonic)" if cfg.align_chroma_to_tonic
                   else "Chroma means (absolute pitch, bin 0 = C)")
axes[1].set_xlabel("Chroma bin")

plt.tight_layout()
plt.show()
